In [ ]:
import pandas as pd
import numpy as np

# Create a DataFrame of Integers with some missing values
df = pd.DataFrame({
    'A' : [1, 2, np.nan, 4],
    'B' : [5, np.nan, np.nan, 8],
    'C' : [9, 10, 11, 12]
})
# Fill missing values with mean
df.fillna(df.mean(), inplace=True)

print(df)

INFO | Logging to C:\Users\david\building-data-pipelines\chapter_5\logs\transformation_pipeline.log
StreamHandler None
RotatingFileHandler C:\Users\david\building-data-pipelines\chapter_5\logs\transformation_pipeline.log
INFO | ROOT says hi
INFO | MODULE logger says hi
INFO | NOTEBOOK logger says hi
          A    B   C
0  1.000000  5.0   9
1  2.000000  6.5  10
2  2.333333  6.5  11
3  4.000000  8.0  12


In [ ]:
import pandas as pd
df_crashes = pd.read_csv("/users/david/building-data-pipelines/chapter_5/data/traffic_crashes.csv")
df_crashes.head()

,crash_record_id,rd_no,crash_date_est_i,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,crash_month,latitude,longitude,location
0,530411c8611eb0ccb9b25f16b2955cd21761fa1928dcaa...,JE494048,NaN,2021-12-31T14:00:00.000,35,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,TURNING,...,0.0,0.0,2.0,0.0,14,6,12,41.794850,-87.767280,POINT (-87.767280356289 41.794849958048)
1,305b06235b250aa0029c07313c84f969f4bc13c1cc3715...,JE494008,NaN,2021-12-31T14:00:00.000,30,TRAFFIC SIGNAL,UNKNOWN,CLEAR,DUSK,TURNING,...,0.0,0.0,2.0,0.0,14,6,12,41.881271,-87.686536,POINT (-87.686535940171 41.881270504288)
2,444221c2a9bc82fc4f301062ab22b482d7d661cf88fcdf...,JE494016,Y,2021-12-31T13:56:00.000,10,OTHER,NO CONTROLS,CLEAR,DAYLIGHT,SIDESWIPE SAME DIRECTION,...,0.0,0.0,2.0,0.0,13,6,12,41.722941,-87.662863,POINT (-87.662862871273 41.72294121821)
3,4603435fbb4ef5d45c0d805c3e9aa5558a311a140a737e...,JE494049,NaN,2021-12-31T13:46:00.000,30,NO CONTROLS,NO CONTROLS,RAIN,DAYLIGHT,PEDALCYCLIST,...,1.0,0.0,2.0,0.0,13,6,12,41.766336,-87.578270,POINT (-87.578269718478 41.766335621716)
4,db62bb4534d0dae57112ea3ff8d50193784aaa732ed58d...,JE494000,NaN,2021-12-31T13:45:00.000,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DAYLIGHT,SIDESWIPE SAME DIRECTION,...,0.0,0.0,2.0,0.0,13,6,12,41.751150,-87.607802,POINT (-87.607802036151 41.7511501753)


In [ ]:
df_crashes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 49 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   crash_record_id                1000 non-null   object 
 1   rd_no                          1000 non-null   object 
 2   crash_date_est_i               69 non-null     object 
 3   crash_date                     1000 non-null   object 
 4   posted_speed_limit             1000 non-null   int64  
 5   traffic_control_device         1000 non-null   object 
 6   device_condition               1000 non-null   object 
 7   weather_condition              1000 non-null   object 
 8   lighting_condition             1000 non-null   object 
 9   first_crash_type               1000 non-null   object 
 10  trafficway_type                1000 non-null   object 
 11  lane_cnt                       1 non-null      float64
 12  alignment                      1000 non-null   ob

In [ ]:
df_crashes.isnull().sum()

crash_record_id                     0
rd_no                               0
crash_date_est_i                  931
crash_date                          0
posted_speed_limit                  0
traffic_control_device              0
device_condition                    0
weather_condition                   0
lighting_condition                  0
first_crash_type                    0
trafficway_type                     0
lane_cnt                          999
alignment                           0
roadway_surface_cond                0
road_defect                         0
report_type                        24
crash_type                          0
intersection_related_i            729
private_property_i                955
hit_and_run_i                     680
damage                              0
date_police_notified                0
prim_contributory_cause             0
sec_contributory_cause              0
street_no                           0
street_direction                    0
street_name 

In [ ]:
df_crashes.dropna(axis='columns', how='all', inplace=True)

In [ ]:
df_crashes = df_crashes.dropna(axis='index', thresh=2, inplace=False)

In [ ]:
print(df_crashes['report_type'].unique())

['ON SCENE' 'NOT ON SCENE (DESK REPORT)' nan]


In [ ]:
df_crashes = df_crashes.fillna(value={'report_type': 'ON SCENE'})

In [ ]:
print(df_crashes['report_type'].unique())

['ON SCENE' 'NOT ON SCENE (DESK REPORT)']


In [ ]:
df_vehicles = pd.read_csv("/users/david/building-data-pipelines/chapter_5/data/traffic_crash_vehicle.csv")

In [ ]:
df = df_crashes.merge(df_vehicles, how='left', on='crash_record_id', suffixes=('_left', '_right'))

In [ ]:
df_agg = df.groupby('vehicle_type').agg({'crash_record_id': 'count'}).reset_index()
df_agg

,vehicle_type,crash_record_id
0,BUS OVER 15 PASS.,5
1,MOPED OR MOTORIZED BICYCLE,1
2,OTHER,20
3,OTHER VEHICLE WITH TRAILER,1
4,PASSENGER,633
5,PICKUP,33
6,SINGLE UNIT TRUCK WITH TRAILER,2
7,SPORT UTILITY VEHICLE (SUV),138
8,TRACTOR W/ SEMI-TRAILER,5
9,TRACTOR W/O SEMI-TRAILER,2


In [ ]:
number_of_passenger_cars_involved = df_agg[df_agg['vehicle_type'] == 'PASSENGER']['crash_record_id'].array[0]
number_of_passenger_cars_involved

633

In [ ]:
vehicle_mapping = {'vehicle_type' : 'vehichletypes'}
df_agg = df_agg.rename(columns = vehicle_mapping)
df_agg


,vehichletypes,crash_record_id
0,BUS OVER 15 PASS.,5
1,MOPED OR MOTORIZED BICYCLE,1
2,OTHER,20
3,OTHER VEHICLE WITH TRAILER,1
4,PASSENGER,633
5,PICKUP,33
6,SINGLE UNIT TRUCK WITH TRAILER,2
7,SPORT UTILITY VEHICLE (SUV),138
8,TRACTOR W/ SEMI-TRAILER,5
9,TRACTOR W/O SEMI-TRAILER,2


In [ ]:
def get_transformed_data(crash_file, vehicle_file):
    # import data
    df_crashes = pd.read_csv(f"/users/david/building-data-pipelines/chapter_5/data/{crash_file}")
    df_vehicles = pd.read_csv(f"/users/david/building-data-pipelines/chapter_5/data/{vehicle_file}")

    # remove specified missing values
    under_threshold_removed = df_crashes.dropna(axis='index', thresh=2, inplace=False) 
    under_threshold_rows = df_crashes[~df_crashes.index.isin(under_threshold_removed.index)] 
        
    # merge crashes and vehicles
    df = df_crashes.merge(df_vehicles, how='left', on='crash_record_id', suffixes=('_left', '_right'))
    df_agg = df.groupby('vehicle_type').agg({'crash_record_id': 'count'}).reset_index()

    # transform column names for output data
    vehicle_mapping = {'vehicle_type': 'vehicletypes'}
    df_agg = df_agg.rename(columns=vehicle_mapping)

    return df_agg
    



In [ ]:
get_transformed_data('traffic_crashes.csv', 'traffic_crash_vehicle.csv')

,vehicletypes,crash_record_id
0,BUS OVER 15 PASS.,5
1,MOPED OR MOTORIZED BICYCLE,1
2,OTHER,20
3,OTHER VEHICLE WITH TRAILER,1
4,PASSENGER,633
5,PICKUP,33
6,SINGLE UNIT TRUCK WITH TRAILER,2
7,SPORT UTILITY VEHICLE (SUV),138
8,TRACTOR W/ SEMI-TRAILER,5
9,TRACTOR W/O SEMI-TRAILER,2


In [ ]:
# Read data from data source
def read_datasources(sourcename):
    df = pd.read_csv(f"/users/david/building-data-pipelines/chapter_5/data/{sourcename}")
    return df

In [ ]:
# Drop rows with null values
def drop_rows_with_null_values(df): 
    cleaned = df.dropna(axis='index', thresh=2)  
    return cleaned 

In [ ]:
# Fill missing values
def fill_missing_values(df):
    df = df.fillna(value={'report_type': 'ON SCENE'})
    return df

In [ ]:
# Merge DataFrames
def merge_dataframes(df_vehicles, df_crashes):
    df = df_crashes.merge(df_vehicles, how='left', on='crash_record_id', suffixes=('_left', '_right'))
    return df

In [ ]:
# Rename columns
def rename_columns(df):
    vehicle_mapping = {'vehicle_type' : 'vehicletypes'}
    df = df.rename(columns=vehicle_mapping)
    return df

In [416]:
def read_data_pipeline(crash_file, vehicle_file):
    df_crash = pd.DataFrame()
    df_vehicle = pd.DataFrame()
    logger.info("Running read_data_pipeline")
    try:
        df_crash = read_datasources(crash_file)
        df_vehicle = read_datasources(vehicle_file)
        status = "<OK>"
        logger.info(f"{status} - read_data_pipeline finished successfully")
    except Exception as e:
        status = "<FAILED>"
        logger.error(f"{status} - read_data_pipeline failed with error: {e}")
    finally:
        return df_crash, df_vehicle, status


DEBUG | 
*** MESSAGE TYPE:execute_request***
DEBUG |    Content: {'silent': False, 'store_history': True, 'user_expressions': {}, 'allow_stdin': True, 'stop_on_error': False, 'code': 'def read_data_pipeline(crash_file, vehicle_file):\n    df_crash = pd.DataFrame()\n    df_vehicle = pd.DataFrame()\n    logger.info("Running read_data_pipeline")\n    try:\n        df_crash = read_datasources(crash_file)\n        df_vehicle = read_datasources(vehicle_file)\n        status = "<OK>"\n        logger.info(f"{status} - read_data_pipeline finished successfully")\n    except Exception as e:\n        status = "<FAILED>"\n        logger.error(f"{status} - read_data_pipeline failed with error: {e}")\n    finally:\n        return df_crash, df_vehicle, status\n'}
   --->
   
DEBUG | execute_request: {'header': {'date': datetime.datetime(2025, 11, 7, 15, 42, 36, 557000, tzinfo=tzutc()), 'msg_id': '36fbb391-d877-4aac-8f70-03e3e70ea711', 'msg_type': 'execute_request', 'session': '06528217-efab-45a9-8df2-

In [425]:
def drop_rows_with_null_values_pipeline(df_crash, df_vehicle):
    logger.info("Running drop_rows_with_null_values_pipeline")
    try:
        df_crash = drop_rows_with_null_values(df_crash)
        df_vehicle = drop_rows_with_null_values(df_vehicles)
        status = "<OK>"
        logger.info(f"{status} - drop_rows_with_null_values finished successfully")
    except Exception as e:
        status = "<FAILED>"
        logger.error(f"{status} - drop_rows_with_null_values failed with error: {e}")
    finally:
        return df_crash, df_vehicle, status


DEBUG | 
*** MESSAGE TYPE:execute_request***
DEBUG |    Content: {'silent': False, 'store_history': True, 'user_expressions': {}, 'allow_stdin': True, 'stop_on_error': False, 'code': 'def drop_rows_with_null_values_pipeline(df_crash, df_vehicle):\n    logger.info("Running drop_rows_with_null_values_pipeline")\n    try:\n        df_crash = drop_rows_with_null_values(df_crash)\n        df_vehicle = drop_rows_with_null_values(df_vehicles)\n        status = "<OK>"\n        logger.info(f"{status} - drop_rows_with_null_values finished successfully")\n    except Exception as e:\n        status = "<FAILED>"\n        logger.error(f"{status} - drop_rows_with_null_values failed with error: {e}")\n    finally:\n        return df_crash, df_vehicle, status\n'}
   --->
   
DEBUG | execute_request: {'header': {'date': datetime.datetime(2025, 11, 7, 15, 42, 43, 768000, tzinfo=tzutc()), 'msg_id': 'da8d2cd2-7780-45a6-9107-90ade68f5ed9', 'msg_type': 'execute_request', 'session': '06528217-efab-45a9-8df2-0

In [433]:
def fill_missing_values_pipeline(df_crash, df_vehicle):
    logger.info("Running fill_missing_values_pipeline")
    try:
        df_crash = fill_missing_values(df_crash)
        df_vehicle = fill_missing_values(df_vehicle)
        status = "<OK>"
        logger.info(f"{status} - fill_missing_values finished successfully")
    except Exception as e:
        status = "<FAILED>"
        logger.error(f"{status} - fill_missing_values failed with error: {e}")
    finally:
        return df_crash, df_vehicle, status

DEBUG | 
*** MESSAGE TYPE:execute_request***
DEBUG |    Content: {'silent': False, 'store_history': True, 'user_expressions': {}, 'allow_stdin': True, 'stop_on_error': False, 'code': 'def fill_missing_values_pipeline(df_crash, df_vehicle):\n    logger.info("Running fill_missing_values_pipeline")\n    try:\n        df_crash = fill_missing_values(df_crash)\n        df_vehicle = fill_missing_values(df_vehicle)\n        status = "<OK>"\n        logger.info(f"{status} - fill_missing_values finished successfully")\n    except Exception as e:\n        status = "<FAILED>"\n        logger.error(f"{status} - fill_missing_values failed with error: {e}")\n    finally:\n        return df_crash, df_vehicle, status'}
   --->
   
DEBUG | execute_request: {'header': {'date': datetime.datetime(2025, 11, 7, 15, 42, 48, 150000, tzinfo=tzutc()), 'msg_id': 'eb416116-1276-480b-acf4-38c6783bbec4', 'msg_type': 'execute_request', 'session': '06528217-efab-45a9-8df2-0c5cbd856db5', 'username': '91502926-1afe-4147

In [440]:
def merge_dataframes_pipeline(df_crash, df_vehicle):
    logger.info("Running merge_dataframes_pipeline")
    try:
        df_agg = merge_dataframes(df_vehicles, df_crash)
        status = "<OK>"
        logger.info(f"{status} - merge_dataframes finished successfully")
    except Exception as e:
        status = "<FAILED>"
        logger.error(f"{status} - merge_dataframes failed with error: {e}")
    finally:
        return df_agg, status

DEBUG | 
*** MESSAGE TYPE:execute_request***
DEBUG |    Content: {'silent': False, 'store_history': True, 'user_expressions': {}, 'allow_stdin': True, 'stop_on_error': False, 'code': 'def merge_dataframes_pipeline(df_crash, df_vehicle):\n    logger.info("Running merge_dataframes_pipeline")\n    try:\n        df_agg = merge_dataframes(df_vehicles, df_crash)\n        status = "<OK>"\n        logger.info(f"{status} - merge_dataframes finished successfully")\n    except Exception as e:\n        status = "<FAILED>"\n        logger.error(f"{status} - merge_dataframes failed with error: {e}")\n    finally:\n        return df_agg, status'}
   --->
   
DEBUG | execute_request: {'header': {'date': datetime.datetime(2025, 11, 7, 15, 42, 52, 732000, tzinfo=tzutc()), 'msg_id': '6332759a-4d23-4188-8b5a-6d26813850c6', 'msg_type': 'execute_request', 'session': '06528217-efab-45a9-8df2-0c5cbd856db5', 'username': '91502926-1afe-4147-9a08-bc238d9d82a5', 'version': '5.2'}, 'msg_id': '6332759a-4d23-4188-8b

In [446]:
def format_dataframes_pipeline(df_agg):
    logger.info("Running format_dataframes_pipeline")
    try:
        df_output = rename_columns(df_agg)
        status = "<OK>"
        logger.info(f"{status} - rename_columns finished successfully")
    except Exception as e:
        status = "<FAILED>"
        logger.error(f"{status} - rename_columns failed with error: {e}")
    finally:
        return df_output, status


DEBUG | 
*** MESSAGE TYPE:execute_request***
DEBUG |    Content: {'silent': False, 'store_history': True, 'user_expressions': {}, 'allow_stdin': True, 'stop_on_error': False, 'code': 'def format_dataframes_pipeline(df_agg):\n    logger.info("Running format_dataframes_pipeline")\n    try:\n        df_output = rename_columns(df_agg)\n        status = "<OK>"\n        logger.info(f"{status} - rename_columns finished successfully")\n    except Exception as e:\n        status = "<FAILED>"\n        logger.error(f"{status} - rename_columns failed with error: {e}")\n    finally:\n        return df_output, status\n'}
   --->
   
DEBUG | execute_request: {'header': {'date': datetime.datetime(2025, 11, 7, 15, 42, 56, 919000, tzinfo=tzutc()), 'msg_id': '58cfef8d-effc-43b0-9135-46444660c522', 'msg_type': 'execute_request', 'session': '06528217-efab-45a9-8df2-0c5cbd856db5', 'username': '91502926-1afe-4147-9a08-bc238d9d82a5', 'version': '5.2'}, 'msg_id': '58cfef8d-effc-43b0-9135-46444660c522', 'msg_ty

In [447]:
# Define input data
crash_data_file = "traffic_crashes.csv"
vehicle_data_file = "traffic_crash_vehicle.csv"

# Read Data Pipeline
df_crash, df_vehicle, status = read_data_pipeline(crash_data_file, vehicle_data_file)

# Drop Nulls
df_crash, df_vehicle, status = drop_rows_with_null_values_pipeline(df_crash, df_vehicle)

# Fill in Missing Values
df_crash, df_vehicle, status = fill_missing_values_pipeline(df_crash, df_vehicle)

# Merge DataFrames
df_agg, status = merge_dataframes_pipeline(df_crash, df_vehicle)

# Rename Columns
df_output, status = format_dataframes_pipeline(df_agg)





DEBUG | 
*** MESSAGE TYPE:execute_request***
DEBUG |    Content: {'silent': False, 'store_history': True, 'user_expressions': {}, 'allow_stdin': True, 'stop_on_error': False, 'code': '# Define input data\ncrash_data_file = "traffic_crashes.csv"\nvehicle_data_file = "traffic_crash_vehicle.csv"\n\n# Read Data Pipeline\ndf_crash, df_vehicle, status = read_data_pipeline(crash_data_file, vehicle_data_file)\n\n# Drop Nulls\ndf_crash, df_vehicle, status = drop_rows_with_null_values_pipeline(df_crash, df_vehicle)\n\n# Fill in Missing Values\ndf_crash, df_vehicle, status = fill_missing_values_pipeline(df_crash, df_vehicle)\n\n# Merge DataFrames\ndf_agg, status = merge_dataframes_pipeline(df_crash, df_vehicle)\n\n# Rename Columns\ndf_output, status = format_dataframes_pipeline(df_agg)\n\n\n\n'}
   --->
   
DEBUG | execute_request: {'header': {'date': datetime.datetime(2025, 11, 7, 15, 42, 56, 944000, tzinfo=tzutc()), 'msg_id': '2070dfff-c757-4f4f-8c07-bca38c039e6b', 'msg_type': 'execute_request'

In [448]:
df_output.head()

DEBUG | 
*** MESSAGE TYPE:execute_request***
DEBUG |    Content: {'silent': False, 'store_history': True, 'user_expressions': {}, 'allow_stdin': True, 'stop_on_error': False, 'code': 'df_output.head()'}
   --->
   
DEBUG | execute_request: {'header': {'date': datetime.datetime(2025, 11, 7, 15, 42, 57, 126000, tzinfo=tzutc()), 'msg_id': '850d35da-749e-4cb9-93e6-87fa25c4e074', 'msg_type': 'execute_request', 'session': '06528217-efab-45a9-8df2-0c5cbd856db5', 'username': '91502926-1afe-4147-9a08-bc238d9d82a5', 'version': '5.2'}, 'msg_id': '850d35da-749e-4cb9-93e6-87fa25c4e074', 'msg_type': 'execute_request', 'parent_header': {}, 'metadata': {'cellId': 'vscode-notebook-cell:/c%3A/Users/david/building-data-pipelines/chapter_5/transformations.ipynb#X41sZmlsZQ%3D%3D'}, 'content': {'silent': False, 'store_history': True, 'user_expressions': {}, 'allow_stdin': True, 'stop_on_error': False, 'code': 'df_output.head()'}, 'buffers': []}


,crash_record_id,rd_no_left,crash_date_est_i,crash_date_left,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,...,trailer1_length,trailer2_length,total_vehicle_length,axle_cnt,vehicle_config,cargo_body_type,load_type,hazmat_out_of_service_i,mcs_out_of_service_i,hazmat_class
0,530411c8611eb0ccb9b25f16b2955cd21761fa1928dcaa...,JE494048,NaN,2021-12-31T14:00:00.000,35,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,TURNING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,530411c8611eb0ccb9b25f16b2955cd21761fa1928dcaa...,JE494048,NaN,2021-12-31T14:00:00.000,35,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,TURNING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,305b06235b250aa0029c07313c84f969f4bc13c1cc3715...,JE494008,NaN,2021-12-31T14:00:00.000,30,TRAFFIC SIGNAL,UNKNOWN,CLEAR,DUSK,TURNING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,305b06235b250aa0029c07313c84f969f4bc13c1cc3715...,JE494008,NaN,2021-12-31T14:00:00.000,30,TRAFFIC SIGNAL,UNKNOWN,CLEAR,DUSK,TURNING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,305b06235b250aa0029c07313c84f969f4bc13c1cc3715...,JE494008,NaN,2021-12-31T14:00:00.000,30,TRAFFIC SIGNAL,UNKNOWN,CLEAR,DUSK,TURNING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


DEBUG | {'header': {'msg_id': 'ff950be0-391df8cf483f38cc2b4d2e8e_45932_3928', 'msg_type': 'execute_reply', 'username': 'username', 'session': 'ff950be0-391df8cf483f38cc2b4d2e8e', 'date': datetime.datetime(2025, 11, 7, 15, 42, 57, 179213, tzinfo=datetime.timezone.utc), 'version': '5.3'}, 'msg_id': 'ff950be0-391df8cf483f38cc2b4d2e8e_45932_3928', 'msg_type': 'execute_reply', 'parent_header': {'date': datetime.datetime(2025, 11, 7, 15, 42, 57, 126000, tzinfo=tzutc()), 'msg_id': '850d35da-749e-4cb9-93e6-87fa25c4e074', 'msg_type': 'execute_request', 'session': '06528217-efab-45a9-8df2-0c5cbd856db5', 'username': '91502926-1afe-4147-9a08-bc238d9d82a5', 'version': '5.2'}, 'content': {'status': 'ok', 'execution_count': 448, 'user_expressions': {}, 'payload': []}, 'metadata': {'started': datetime.datetime(2025, 11, 7, 15, 42, 57, 140820, tzinfo=datetime.timezone.utc), 'dependencies_met': True, 'engine': 'c64658ef-4e14-47db-acd3-749433388b41', 'status': 'ok'}, 'tracker': <zmq.sugar.tracker.Message

In [454]:
READING_CRASH_DATA_PIPELINE = "<NOT_EXECUTED>"
DROPPING_ROW_WITH_NULL_PIPELINE = "<NOT_EXECUTED>"
FILLING_MISSING_VALUE_PIPELINE = "<NOT_EXECUTED>"
MERGE_DATA_FRAME_PIPELINE = "<NOT_EXECUTED>"
FORMAT_DATAFRAME_PIPELINE = "<NOT_EXECUTED>"

In [ ]:
# In the first cell of transformations.ipynb
%run -i bootstrap.py  # loads the bootstrap() function

logger, logpath = bootstrap(unique=True, level="INFO")  # or level="DEBUG"
logger.info(f"Ready. Writing to: {logpath}")

# Anywhere in the notebook
from config.log_config import get_logger
step_log = get_logger("transformation.pipeline")
step_log.info("Starting transformation pipeline")

df_crash, df_vehicle, READING_CRASH_DATA_PIPELINE = read_data_pipeline("traffic_crashes.csv", "traffic_crash_vehicle.csv")

if READING_CRASH_DATA_PIPELINE == "<OK>":
    df_crash, df_vehicle, DROPPING_ROW_WITH_NULL_PIPELINE = drop_rows_with_null_values_pipeline(df_crash, df_vehicle)

if DROPPING_ROW_WITH_NULL_PIPELINE == "<OK>":
    df_crash, df_vehicle, FILLING_MISSING_VALUE_PIPELINE = fill_missing_values_pipeline(df_crash, df_vehicle)

if FILLING_MISSING_VALUE_PIPELINE == "<OK>":
    df_agg, MERGE_DATA_FRAME_PIPELINE = merge_dataframes_pipeline(df_crash, df_vehicle)

if MERGE_DATA_FRAME_PIPELINE == "<OK>":
    df_output, FORMAT_DATAFRAME_PIPELINE = format_dataframes_pipeline(df_agg)

INFO | Logging to c:\Users\david\building-data-pipelines\chapter_5\logs\etl_DEV_20251107-105052_45932.log
INFO | Notebook logging initialized
Initialized logging to: c:\Users\david\building-data-pipelines\chapter_5\logs\etl_DEV_20251107-105052_45932.log
INFO | bootstrap.py executed as a script
INFO | Logging to c:\Users\david\building-data-pipelines\chapter_5\logs\etl_DEV_20251107-105052_45932.log
INFO | Notebook logging initialized
INFO | Ready. Writing to: c:\Users\david\building-data-pipelines\chapter_5\logs\etl_DEV_20251107-105052_45932.log
INFO | Starting transformation pipeline
INFO | Running read_data_pipeline
INFO | <OK> - read_data_pipeline finished successfully
INFO | Running drop_rows_with_null_values_pipeline
INFO | <OK> - drop_rows_with_null_values finished successfully
INFO | Running fill_missing_values_pipeline
INFO | <OK> - fill_missing_values finished successfully
INFO | Running merge_dataframes_pipeline
INFO | <OK> - merge_dataframes finished successfully
INFO | Runni